In [0]:
%sql 
create schema if not exists databrickslearning.bronze;

In [0]:
source_path="abfss://data@databrickslearningsa.dfs.core.windows.net/staging/diagnosis/"
checkpoint_path="abfss://data@databrickslearningsa.dfs.core.windows.net/bronze/diagnosis_raw/checkpoint/"
schema_location="abfss://data@databrickslearningsa.dfs.core.windows.net/bronze/diagnosis_raw/schema/"

#AutoLoader Read
df = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "csv")
  .option("header", True)
  .option("cloudFiles.maxFilesPerTrigger", 1)
  .option("cloudFiles.schemaLocation", schema_location)
  .load(source_path)
)
#Write Stream
(
  df.drop("_rescued_data")
    .writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable("databrickslearning.bronze.diagnosis_raw")
)

In [0]:
%sql

select * from databrickslearning.bronze.diagnosis_raw